# TePMA — Data generation & evaluation (GPU runtime)

**Runtime → Change runtime type → T4 GPU.** Not TPU: this notebook runs the Qwen3-8B
*teacher* through Ollama, and Ollama has no TPU backend — on a TPU runtime the 8B falls
back to CPU and generation becomes slower than a laptop.

This notebook covers three of the four pipeline steps. Training is the other notebook.

```
  1. GENERATE  (here)  Qwen3-8B invents Indian interviews in en/hi/pa   -> raw/*.json
  2. BUILD     (here)  raw samples -> train.jsonl / eval.jsonl
  3. TRAIN     (TePMA_finetune_colab.ipynb)  QLoRA -> tepma GGUF
  4. EVALUATE  (here)  score tepma against qwen3:8b on held-out interviews
```

Everything is written to **Google Drive**, so a Colab disconnect costs you nothing —
just re-run and generation resumes where it stopped.

---
## Step 0 — Mount Drive and clone the repo

Set `REPO` to your private GitHub repo. For a private repo, create a fine-grained personal
access token with read-only Contents permission and paste it when prompted — it is read
with `getpass`, so it never lands in the notebook output or in Drive.

In [ ]:
import os, getpass, subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO = 'your-username/TePMA'      # <-- edit this
WORK = Path('/content/drive/MyDrive/tepma')
WORK.mkdir(parents=True, exist_ok=True)

# Outputs live on Drive; the code is disposable and re-cloned each session.
os.environ['TEPMA_RAW_DIR']  = str(WORK / 'raw')
os.environ['TEPMA_DATA_DIR'] = str(WORK / 'data')

if not Path('/content/TePMA').exists():
    token = getpass.getpass('GitHub token (blank if the repo is public): ').strip()
    url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
    subprocess.run(['git', 'clone', '--depth', '1', url, '/content/TePMA'], check=True)
else:
    subprocess.run(['git', '-C', '/content/TePMA', 'pull'], check=True)

%cd /content/TePMA
print('raw ->', os.environ['TEPMA_RAW_DIR'])
print('existing samples:', len(list((WORK / 'raw').glob('sample_*.json'))) if (WORK / 'raw').exists() else 0)

---
## Step 1 — Dependencies

`generate_data.py` imports `engines.py`, which pulls in the audio stack at import time.
We do **not** need Whisper or Kokoro here (they load lazily, and this notebook never calls
them) — only the handful of packages imported at module level, plus FastAPI because
`build_dataset.py` reads `INTERVIEWER_PROMPT` out of `routes_interview.py`.

In [ ]:
%%capture
!pip install httpx python-dotenv soundfile fastapi

---
## Step 2 — Install Ollama and pull the teacher

`OLLAMA_NUM_PARALLEL` is the important setting: it lets the server handle several requests
at once. A single sequential request leaves most of the T4 idle — the GPU is waiting on one
token at a time. Eight in flight is roughly a 5–6× throughput win on the same hardware.

The 8B pull is ~5 GB and takes a few minutes.

In [ ]:
import os, subprocess, time, httpx

!curl -fsSL https://ollama.com/install.sh | sh

env = {**os.environ, 'OLLAMA_NUM_PARALLEL': '8', 'OLLAMA_FLASH_ATTENTION': '1'}
subprocess.Popen(['ollama', 'serve'], env=env,
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(60):
    try:
        if httpx.get('http://127.0.0.1:11434/api/version', timeout=2).status_code == 200:
            print('ollama up'); break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('ollama did not start')

!ollama pull qwen3:8b
!nvidia-smi --query-gpu=name,memory.total --format=csv

---
## Step 3 — Generate the training data

Each sample costs two teacher calls: invent a full interview transcript, then extract the
gold profile from it with the **exact production prompt and schema**. That second call is
the point — the student learns to reproduce the teacher's extraction, so the labels have to
come from the same code path the app runs.

Samples that the teacher botched are dropped by the quality gate in `make_sample` rather
than saved: a bad label teaches the student to be bad in exactly that way.

**Budget roughly 20–30 seconds per sample at concurrency 8 on a T4**, so ~200 samples per
session with room to spare. If the runtime dies, re-run this cell — completed samples on
Drive are skipped automatically.

In [ ]:
!python finetune/generate_data.py --count 200 --concurrency 8

### Check the language mix before you train on it

The mix is sampled randomly, so it drifts from the intended 5:3:2:3 split on small runs. If
Hindi or Punjabi is badly under-represented here, the model will answer those languages in
English — generate more before training rather than discovering it after.

In [ ]:
import json, collections, os
from pathlib import Path

raw = Path(os.environ['TEPMA_RAW_DIR'])
samples = [json.loads(p.read_text()) for p in sorted(raw.glob('sample_*.json'))]
mix = collections.Counter(s['lang'] for s in samples)
print(f'{len(samples)} samples')
for lang, n in mix.most_common():
    print(f'  {lang:6} {n:4}  {n/len(samples):5.1%}')

print('\n--- one Punjabi/Hindi sample, spot-check the quality by eye ---')
for s in samples:
    if s['lang'] in ('pa', 'hi'):
        for t in s['transcript'][:4]:
            print(f"{t['speaker']:11}: {t['text']}")
        break

---
## Step 4 — Build the datasets

Turns the raw samples into chat-format JSONL, holding out 10% of *interviews* (not rows) so
no transcript appears in both train and eval. Produces three files on Drive:

- `train.jsonl` / `eval.jsonl` — for the training notebook
- `eval_extraction.jsonl` — held-out transcripts + gold profiles, used by `eval_model.py`

In [ ]:
!python finetune/build_dataset.py

---
## Step 5 — Measure the baselines *before* training

This is the step that tells you whether fine-tuning is even the right call, and it is worth
the twenty minutes. Three numbers:

| Model | What its score means |
|---|---|
| `qwen3:8b` | the target to match — what your app scores today |
| `qwen3:4b` | if this lands near the 8B **untuned**, ship it and skip fine-tuning |
| `qwen3:1.7b` | the gap the fine-tune has to close. If it collapses, fine-tune 4B instead |

Ignore the timings printed here — a Colab T4 is not your kiosk Mac. Only the accuracy
numbers transfer.

In [ ]:
!ollama pull qwen3:4b && ollama pull qwen3:1.7b

for model in ['qwen3:8b', 'qwen3:4b', 'qwen3:1.7b']:
    print(f'\n{"#" * 60}\n# {model}\n{"#" * 60}')
    !python finetune/eval_model.py --model {model}

---
## Step 6 — After training: score the fine-tuned model

Come back here once the training notebook has produced `tepma.gguf` on Drive. Same eval set,
same scorer, so the number is directly comparable to the baselines above.

**Success:** `tepma` lands within a few points of `qwen3:8b` on OVERALL, and does not
regress on `no_hallucination` — inventing a job the candidate never mentioned is the one
failure a résumé tool cannot ship with.

In [ ]:
from pathlib import Path
import os

gguf = Path('/content/drive/MyDrive/tepma/tepma.gguf')
assert gguf.exists(), f'{gguf} not found — run the training notebook first'

Path('/content/Modelfile').write_text(f'FROM {gguf}\n')
!ollama create tepma -f /content/Modelfile
!python finetune/eval_model.py --model tepma

---
## Last step — and it is not on Colab

Accuracy is settled above. **Latency is not**, and latency is the entire reason for this
project: the number users feel is how long they sit in silence waiting for the interviewer's
next question on the kiosk Mac.

Download `tepma.gguf` from Drive, then on the Mac:

```bash
echo 'FROM ./tepma.gguf' > Modelfile && ollama create tepma -f Modelfile
.venv/bin/python finetune/eval_model.py --model tepma
.venv/bin/python finetune/eval_model.py --model qwen3:8b
```

Compare the `interview reply` medians. If `tepma` is several times faster at comparable
accuracy, set `LLM_MODEL=tepma` in `.env` and you are done.